In [1]:
# Fetch the SRGAN code
!git clone https://github.com/Lornatang/SRGAN-PyTorch.git

# Move into the folder
%cd SRGAN-PyTorch

# Note: We are skipping the requirements.txt because Kaggle 
# already has a perfectly optimized PyTorch environment built-in!

Cloning into 'SRGAN-PyTorch'...
remote: Enumerating objects: 3631, done.
remote: Counting objects: 100% (920/920), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 3631 (delta 690), reused 810 (delta 633), pack-reused 2711 (from 1)
Receiving objects: 100% (3631/3631), 2.98 MiB | 15.63 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/kaggle/working/SRGAN-PyTorch


In [2]:
import os
import yaml

config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# 1. 90-Epoch Lock
cfg['TRAIN']['HYP']['EPOCHS'] = 90
cfg['TRAIN']['LR_SCHEDULER']['MILESTONES'] = [45, 68]

# 2. Map SAR Datasets
base_path = "/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset"
cfg['TRAIN']['DATASET']['TRAIN_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TRAIN']['DATASET']['TRAIN_LR_IMAGES_DIR'] = f"{base_path}/train_LR"
cfg['TEST']['DATASET']['PAIRED_TEST_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 3. Start from Phase 1 Weights
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = "/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar"
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_D_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = ""

# 4. Fix Kaggle GPU compiled flags
cfg['MODEL']['G']['COMPILED'] = False
cfg['MODEL']['D']['COMPILED'] = False
cfg['MODEL']['EMA']['COMPILED'] = False

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

# --- THE HOT-PATCHES ---
utils_path = 'utils.py'
with open(utils_path, 'r') as f:
    utils_code = f.read()

# Fix the Scheduler Resume Bug
utils_code = utils_code.replace(
    'scheduler.load_state_dict(checkpoint["scheduler"])',
    'pass # [HOTFIX] Bypassed missing scheduler!'
)

# Fix the 20GB Disk Crash Bug (The Rolling Backup Patch)
rolling_backup_code = """torch.save(state_dict, checkpoint_path)
    try:
        import os
        if "epoch_" in checkpoint_path:
            ep_num = int(checkpoint_path.split("epoch_")[-1].split(".")[0])
            prev_file = checkpoint_path.replace(f"epoch_{ep_num}", f"epoch_{ep_num-1}")
            if os.path.exists(prev_file): os.remove(prev_file)
    except: pass"""

utils_code = utils_code.replace(
    'torch.save(state_dict, checkpoint_path)',
    rolling_backup_code
)

with open(utils_path, 'w') as f:
    f.write(utils_code)

print("✅ Phase 1 weights mapped. 90 Epochs locked.")
print("✅ Source code patched: Rolling backup activated to prevent disk crash!")

✅ Phase 1 weights mapped. 90 Epochs locked.
✅ Source code patched: Rolling backup activated to prevent disk crash!


In [3]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-17 08:08:10.421313: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773734890.636578      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773734890.708890      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773734891.251536      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773734891.251593      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773734891.251599      55 computation_placer.cc:177] computation placer alr